# Import & Preprocessing

In [ ]:
!pip install openpyxl lifelines

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
SAVEFIG = "savefig/"

In [ ]:
# Load & Clean Online Retail
xls = pd.ExcelFile('dataset/Online_Retail.xlsx')
df = pd.read_excel(xls, sheet_name=xls.sheet_names[0])
df = df[(df['Quantity'] > 0) & (df['CustomerID'].notna())].copy()

print(f"Data bersih: {len(df):,} baris")

In [ ]:
# Mapping StockCode → nama produk indo
top_158 = df['StockCode'].value_counts().head(158).index.tolist()
with open('dataset/katalog.json') as f:
    katalog = [p['nama_produk'] for p in json.load(f)['produk']]

n = min(len(top_158), len(katalog))
mapping = dict(zip(top_158[:n], katalog[:n]))
df_f = df[df['StockCode'].isin(mapping)].copy()
df_f['StockCode'] = df_f['StockCode'].map(mapping)
df_f['Description'] = df_f['StockCode']
df_f.to_csv('transaksi_sembako_clean.csv', index=False)

print(f"Produk termapping: {df_f['StockCode'].nunique()}")
print(f"Baris tersimpan: {len(df_f):,}")

In [ ]:
# Load clean CSV & setup kolom
df_clean = pd.read_csv("transaksi_sembako_clean.csv")
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean = df_clean.sort_values(['CustomerID', 'StockCode', 'InvoiceDate']).reset_index(drop=True)
df_clean['LineTotal']   = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['Month']       = df_clean['InvoiceDate'].dt.month
df_clean['DayOfWeek']   = df_clean['InvoiceDate'].dt.dayofweek
df_clean['Hour']        = df_clean['InvoiceDate'].dt.hour

print(f"Shape: {df_clean.shape}")
print(f"Periode: {df_clean['InvoiceDate'].min()} — {df_clean['InvoiceDate'].max()}")

# EDA (Exploratory Data Analysis)

In [ ]:
df_clean.info()

missing = df_clean.isna().sum()
print("\nMissing:", missing[missing > 0].to_dict() if missing.any() else "Tidak ada")
print(f"Duplikat: {df_clean.duplicated().sum()}")
print(f"Produk unik: {df_clean['StockCode'].nunique()}  |  Pelanggan: {df_clean['CustomerID'].nunique()}  |  Invoice: {df_clean['InvoiceNo'].nunique()}")
display(df_clean[['Quantity', 'UnitPrice', 'LineTotal']].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df_clean['Quantity'], bins=60, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribusi Quantity per Item')
axes[0, 0].set_xlim(0, df_clean['Quantity'].quantile(0.99))

sns.histplot(df_clean['UnitPrice'], bins=60, ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Distribusi UnitPrice')
axes[0, 1].set_xlim(0, df_clean['UnitPrice'].quantile(0.99))

sns.boxplot(x=df_clean['Quantity'], ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('Boxplot Quantity')

sns.boxplot(x=df_clean['UnitPrice'], ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Boxplot UnitPrice')

plt.tight_layout()
plt.savefig(SAVEFIG + "eda_distribusi.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_produk = df_clean['StockCode'].value_counts().head(10)
sns.barplot(x=top_produk.values, y=top_produk.index, hue=top_produk.index, ax=axes[0], palette='viridis', legend=False)
axes[0].set_title('Top 10 Produk')

top_cust = df_clean['CustomerID'].value_counts().head(10)
sns.barplot(x=top_cust.values, y=top_cust.index.astype(int).astype(str), hue=top_cust.index.astype(str), ax=axes[1], palette='magma', legend=False)
axes[1].set_title('Top 10 Pelanggan')

plt.tight_layout()
plt.savefig(SAVEFIG + "eda_top.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_clean.groupby('Month').size().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Transaksi per Bulan')

day_labels = ['Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab', 'Min']
by_day = df_clean.groupby('DayOfWeek').size()
by_day.index = [day_labels[int(i)] for i in by_day.index]
by_day.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Transaksi per Hari')
axes[1].tick_params(axis='x', rotation=0)

df_clean.groupby('Hour').size().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Transaksi per Jam')

plt.tight_layout()
plt.savefig(SAVEFIG + "eda_temporal.png", dpi=150, bbox_inches='tight')
plt.show()

# Feature Engineering

In [ ]:
basket = df_clean.groupby('InvoiceNo').agg(
    basket_size   = ('Quantity',  'sum'),
    basket_unique = ('StockCode', 'nunique'),
    basket_value  = ('LineTotal', 'sum'),
).reset_index()
df_clean = df_clean.merge(basket, on='InvoiceNo', how='left')
display(basket.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(basket['basket_size'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Basket Size (Total Qty)')
axes[0].set_xlim(0, basket['basket_size'].quantile(0.99))

sns.histplot(basket['basket_unique'], bins=30, ax=axes[1], color='coral')
axes[1].set_title('Variasi Produk per Invoice')

sns.histplot(basket['basket_value'], bins=50, ax=axes[2], color='green')
axes[2].set_title('Total Nilai Invoice')
axes[2].set_xlim(0, basket['basket_value'].quantile(0.99))

plt.tight_layout()
plt.savefig(SAVEFIG + "fe_basket.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Agregasi : 1 baris = 1 pelanggan × 1 produk × 1 invoice
trx = df_clean.groupby(
    ['CustomerID', 'StockCode', 'InvoiceNo', 'InvoiceDate'], as_index=False
).agg(
    Quantity       = ('Quantity',      'sum'),
    UnitPrice      = ('UnitPrice',     'mean'),
    LineTotal      = ('LineTotal',     'sum'),
    basket_size    = ('basket_size',   'first'),
    basket_unique  = ('basket_unique', 'first'),
    basket_value   = ('basket_value',  'first'),
    Month          = ('Month',         'first'),
    DayOfWeek      = ('DayOfWeek',     'first'),
    Hour           = ('Hour',          'first'),
).sort_values(['CustomerID', 'StockCode', 'InvoiceDate']).reset_index(drop=True)

grp = trx.groupby(['CustomerID', 'StockCode'], sort=False)

# Waktu
trx['PrevInvoiceDate']  = grp['InvoiceDate'].shift(1)
trx['NextInvoiceDate']  = grp['InvoiceDate'].shift(-1)
trx['DaysSincePrev']    = (trx['InvoiceDate'] - trx['PrevInvoiceDate']).dt.days
trx['Target_Days_To_Next'] = (trx['NextInvoiceDate'] - trx['InvoiceDate']).dt.days

# Frekuensi & statistik riwayat
trx['PurchaseNumber']   = grp.cumcount() + 1

interval = trx['DaysSincePrev']
trx['Avg_Days_Between'] = interval.groupby(
    [trx['CustomerID'], trx['StockCode']]
).transform(lambda x: x.shift().expanding().mean())

print(f"Transaksi (pelanggan × produk): {len(trx):,}")

## Model Survival Analysis (Cox Proportional Hazards)

In [ ]:
#  Event = 1 pembelian berikutnya 
#  Event = 0 pembelian terakhir, belum ada data berikutnya
surv = trx.copy()
study_end = surv['InvoiceDate'].max()

surv['Event'] = surv['NextInvoiceDate'].notna().astype(int)
surv['Duration'] = np.where(
    surv['Event'].eq(1),
    (surv['NextInvoiceDate'] - surv['InvoiceDate']).dt.days,
    (study_end - surv['InvoiceDate']).dt.days
).astype(float)
surv = surv[surv['Duration'] > 0].copy()

# Fitur
surv['Month']            = surv['InvoiceDate'].dt.month
surv['DayOfWeek']        = surv['InvoiceDate'].dt.dayofweek
surv['log_quantity']     = np.log1p(surv['Quantity'].clip(lower=0))
surv['log_line_total']   = np.log1p(surv['LineTotal'].clip(lower=0))
surv['log_basket_size']  = np.log1p(surv['basket_size'].clip(lower=0))
surv['log_basket_value'] = np.log1p(surv['basket_value'].clip(lower=0))
surv['DaysSincePrev']    = surv['DaysSincePrev'].fillna(0)
surv['Avg_Days_Between'] = surv['Avg_Days_Between'].fillna(
    surv['DaysSincePrev'].replace(0, np.nan).median()
).fillna(0)

SURV_FEATURES = [
    'log_quantity', 'UnitPrice', 'log_line_total',
    'log_basket_size', 'basket_unique', 'log_basket_value',
    'PurchaseNumber', 'DaysSincePrev', 'Avg_Days_Between',
    'Month', 'DayOfWeek'
]

surv_df = surv[['Duration', 'Event'] + SURV_FEATURES].replace([np.inf, -np.inf], np.nan).dropna().copy()

print(f"Data survival: {len(surv_df):,}  |  Event rate: {surv_df['Event'].mean():.1%}")

In [ ]:
# Train/Test split
surv_df = surv_df.sort_index()
split = int(len(surv_df) * 0.8)
train_s, test_s = surv_df.iloc[:split].copy(), surv_df.iloc[split:].copy()

cox = CoxPHFitter(penalizer=0.1)
cox.fit(train_s, duration_col='Duration', event_col='Event')

# Evaluasi
risk_test = cox.predict_partial_hazard(test_s)
c_index = concordance_index(test_s['Duration'], -risk_test, test_s['Event'])

print(f"Train: {len(train_s):,}  |  Test: {len(test_s):,}")
print(f"Concordance index: {c_index:.4f}  (0.5 = random, 1.0 = sempurna)")
display(cox.summary[['coef', 'exp(coef)', 'p']].sort_values('p'))

In [ ]:
# Visualisasi evaluasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# C-index bar
axes[0].barh(['CoxPH'], [c_index], color='steelblue')
axes[0].axvline(0.5, color='red', linestyle='--', label='Random baseline')
axes[0].set_xlim(0, 1)
axes[0].set_title(f'Concordance Index = {c_index:.4f}')
axes[0].legend()

# Coefficient forest plot
coefs = cox.summary[['coef']].sort_values('coef')
colors = ['coral' if c < 0 else 'steelblue' for c in coefs['coef']]
axes[1].barh(coefs.index, coefs['coef'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('CoxPH Coefficients')

plt.tight_layout()
plt.savefig(SAVEFIG + "survival_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

# save model & Prediksi

In [ ]:
joblib.dump(cox, "model_coxph_restock.pkl")

metadata = {
    "surv_features": SURV_FEATURES,
    "study_end": str(study_end),
    "c_index": c_index,
    "description": "CoxPH model for predicting time-to-next-purchase"
}
with open("model_coxph_restock_meta.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model disimpan: model_coxph_restock.pkl")
print("Metadata:      model_coxph_restock_meta.json")

In [ ]:
latest_idx = surv.groupby(['CustomerID', 'StockCode'])['InvoiceDate'].idxmax()
latest     = surv.loc[latest_idx].copy()

X_latest   = latest[SURV_FEATURES]
surv_funcs = cox.predict_survival_function(X_latest)

# Median survival time
median_times = []
for i in range(surv_funcs.shape[1]):
    sf = surv_funcs.iloc[:, i]
    hit = sf.values <= 0.5
    median_times.append(sf.index[hit].min() if hit.any() else sf.index.max())

latest['partial_hazard']            = cox.predict_partial_hazard(X_latest)
latest['Pred_Median_Survival_Days'] = np.round(median_times, 1).clip(3, 90)

today = study_end.normalize()
latest['Days_Since_Last_Buy'] = (today - latest['InvoiceDate'].dt.normalize()).dt.days
latest['Pred_Days_Left']      = (
    latest['Pred_Median_Survival_Days'] - latest['Days_Since_Last_Buy']
).clip(lower=0).astype(int)

for h in [7, 14, 30]:
    idx = abs(surv_funcs.index - h).argmin()
    latest[f'Prob_Buy_Within_{h}D'] = (1 - surv_funcs.iloc[idx].values).round(4)

# Output
OUTPUT_COLS = [
    'CustomerID', 'StockCode',
    'DaysSincePrev', 'Avg_Days_Between', 'Pred_Median_Survival_Days',
    'Days_Since_Last_Buy', 'Pred_Days_Left', 'partial_hazard',
    'Prob_Buy_Within_7D', 'Prob_Buy_Within_14D', 'Prob_Buy_Within_30D'
]
pred = latest[OUTPUT_COLS].sort_values(['Pred_Days_Left', 'partial_hazard'], ascending=[True, False])

# Testing Notifikasi Restock

In [ ]:
def should_notify(row):
    return (
        (row['Pred_Days_Left'] <= 3 and row['Prob_Buy_Within_14D'] >= 0.50)
        or (row['Pred_Days_Left'] <= 7 and row['Prob_Buy_Within_7D'] >= 0.60)
    )

def notification_message(row):
    if row['Prob_Buy_Within_7D'] >= 0.60:
        return f"Stok {row['StockCode']} kemungkinan perlu dibeli ulang minggu ini."
    if row['Prob_Buy_Within_14D'] >= 0.50:
        return f"Stok {row['StockCode']} kemungkinan perlu dibeli ulang dalam 2 minggu."
    return f"Pertimbangkan restock {row['StockCode']} segera."

pred['should_notify']        = pred.apply(should_notify, axis=1)
pred['notification_message'] = pred.apply(notification_message, axis=1)

notif = pred[pred['should_notify']].copy()

print(f"Notifikasi : {len(notif)}/{len(pred)} ({len(notif)/len(pred)*100:.1f}%)")

## Input Model

Model `cox` menerima **satu baris** per pasangan `(CustomerID, StockCode)` pada saat transaksi terakhirnya.

| # | Fitur | Tipe | Deskripsi | Cara Mendapatkan |
|---|-------|------|-----------|------------------|
| 1 | `log_quantity` | float | ln(1 + jumlah_item_dibeli) | Dari transaksi saat ini |
| 2 | `UnitPrice` | float | Harga satuan produk | Dari transaksi saat ini |
| 3 | `log_line_total` | float | ln(1 + Quantity × UnitPrice) | Hitung dari transaksi |
| 4 | `log_basket_size` | float | ln(1 + total_item_dalam_invoice) | Agregasi invoice saat ini |
| 5 | `basket_unique` | int | Variasi produk dalam invoice | Hitung dari invoice |
| 6 | `log_basket_value` | float | ln(1 + total_nilai_invoice) | Agregasi invoice saat ini |
| 7 | `PurchaseNumber` | int | Ini pembelian ke-berapa produk ini? | Count dari riwayat |
| 8 | `DaysSincePrev` | float | Hari sejak pembelian sebelumnya | Selisih tanggal dari riwayat |
| 9 | `Avg_Days_Between` | float | Rata-rata jeda historis (hari) | Expanding mean dari riwayat |
| 10 | `Month` | int | Bulan transaksi (1–12) | Dari InvoiceDate |
| 11 | `DayOfWeek` | int | Hari (0=Senin, 6=Minggu) | Dari InvoiceDate |

**Output model:**
- `partial_hazard` risiko relatif (makin tinggi → makin cepat beli ulang)
- `Pred_Median_Survival_Days` estimasi kapan 50% pelanggan sudah beli lagi (3–90 hari)
- `Prob_Buy_Within_7D / 14D / 30D` probabilitas beli ulang dalam 7/14/30 hari
- `Pred_Days_Left` urgensi selisih median survival vs hari sejak beli terakhir (0 = sudah jatuh tempo)

**Setup di flaskApi :**
```python
import joblib, pandas as pd, numpy as np
cox = joblib.load("model_coxph_restock.pkl")

# Setup Data
X_new = pd.DataFrame([{...}], columns=SURV_FEATURES)

# Prediksi survival function
sf = cox.predict_survival_function(X_new)
median = sf.apply(lambda col: col[col <= 0.5].index.min() if (col <= 0.5).any() else col.index.max())
prob_7d  = 1 - sf.iloc[abs(sf.index - 7).argmin()].values[0]
prob_14d = 1 - sf.iloc[abs(sf.index - 14).argmin()].values[0]
prob_30d = 1 - sf.iloc[abs(sf.index - 30).argmin()].values[0]
```


## Alur Trigger Model
```
┌─────────────────────────────────────────────────────────┐
│  1. PELANGGAN CHECKOUT                                  │
│     Transaksi baru masuk ke database                    │
└──────────────────────┬──────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────┐
│  2. EKSTRAK FITUR                                       │
│     Ambil riwayat CustomerID × StockCode dari DB        │
│     Hitung: PurchaseNumber, DaysSincePrev,              │
│             Avg_Days_Between                            │
│     Hitung dari invoice saat ini:                       │
│             log_quantity, UnitPrice, log_line_total,    │
│             log_basket_size, basket_unique,             │
│             log_basket_value, Month, DayOfWeek          │
└──────────────────────┬──────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────┐
│  3. PREDIKSI                                            │
│     cox.predict_survival_function(X_new)                │
│     → Pred_Median_Survival_Days (3–90 hari)            │
│     → Prob_Buy_Within_7D / 14D / 30D                   │
└──────────────────────┬──────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────┐
│  4. SIMPAN PREDIKSI                                     │
│     Simpan ke tabel prediksi dengan kolom:              │
│     customer_id, product_id, predicted_restock_date,    │
│     prob_7d, prob_14d, prob_30d                         │
└──────────────────────┬──────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────┐
│  5. SCHEDULER HARIAN (cron job)                         │
│     Setiap hari jam 06:00:                              │
│     SELECT * FROM predictions                           │
│     WHERE predicted_restock_date - INTERVAL '3 days'    │
│           <= CURRENT_DATE                               │
│       AND prob_14d >= 0.50                              │
└──────────────────────┬──────────────────────────────────┘
                       ▼
┌─────────────────────────────────────────────────────────┐
│  6. KIRIM NOTIFIKASI                                    │
│     "Stok Beras Premium 5 kg kemungkinan perlu          │
│      dibeli ulang minggu ini."                          │
└─────────────────────────────────────────────────────────┘
```

**Contoh :**

| Step | Data |
|------|------|
| Transaksi masuk | CustomerID=17841, StockCode="Beras Premium 5 kg", Qty=2, hari ini |
| Riwayat dari DB | Ini pembelian ke-4, jeda rata-rata 35 hari, terakhir beli 31 hari lalu |
| Prediksi model | `Pred_Median_Survival_Days` = 32 hari |
| Tanggal prediksi restock | hari ini + 32 = 18 Agustus 2026 |
| Scheduler tiap hari | Cek: `18 Agustus - 3 hari <= hari ini`? |
| Kirim notifikasi | Mulai 15 Agustus: "Stok Beras Premium 5 kg kemungkinan perlu dibeli ulang minggu ini." |

**Trigger point :** model dipanggil saat notifikasi membaca hasil prediksi yang sudah disimpan.

### Penjalasan

Model **tidak perlu menerima semua riwayat transaksi**. cukup mengirim 11 fitur input, tetapi 3 fitur historis perlu dihitung dari riwayat pelanggan-produk:

| Fitur historis | Cara hitung | Minimal | Rekomendasi |
|---|---|---:|---:|
| `PurchaseNumber` | Total pembelian pelanggan untuk produk tersebut | 1 transaksi | Semua riwayat / counter tersimpan |
| `DaysSincePrev` | Jarak hari dari transaksi sebelumnya | 2 transaksi | 2 transaksi terakhir |
| `Avg_Days_Between` | Rata-rata jeda antar pembelian | 2 transaksi | 3–6 transaksi terakhir |

Kualitas prediksi berdasarkan jumlah riwayat:

| Jumlah transaksi pelanggan-produk | Kualitas |
|---:|---|
| 1 | Bisa diprediksi, tapi masih fallback/global |
| 2 | Mulai personal |
| 3–5 | Cukup baik untuk notifikasi |
| ≥6 | Lebih stabil / ideal |

**Rekomendasi** simpan semua transaksi di database, tetapi untuk prediksi cukup ambil maksimal **6 transaksi terakhir** per `(CustomerID, StockCode)` untuk menghitung `DaysSincePrev` dan `Avg_Days_Between`. `PurchaseNumber` sebaiknya tetap memakai total pembelian historis, bisa dari `COUNT(*)` atau counter tersimpan.

Jika pelanggan baru pertama kali membeli produk:

```text
PurchaseNumber = 1
DaysSincePrev = 0
Avg_Days_Between = median global dari data training
```

model tetap bisa dipanggil dari transaksi pertama, tetapi akurasi akan meningkat setelah pelanggan punya minimal 3–5 transaksi untuk produk yang sama.